In [1]:
# PHASE 1 - Bootstrap and baseline lock
import os
import sys
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from torch_geometric.data import Data

root = Path('/home/jupyter-1nt23cb058/Capstone')
if not root.exists():
    raise FileNotFoundError(f'Project root not found: {root}')

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from gnn.model import STPIGNN, train_step_amp, masked_data_loss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BASELINE = {
    'test_scaled_mse': 0.0009470369910732621,
    'test_mae_unscaled': 5.080196380615234,
    'test_rmse_unscaled': 6.913161277770996
}

print({'device': str(device), 'baseline': BASELINE})

{'device': 'cuda', 'baseline': {'test_scaled_mse': 0.0009470369910732621, 'test_mae_unscaled': 5.080196380615234, 'test_rmse_unscaled': 6.913161277770996}}


In [2]:
# PHASE 2a - Config and baseline lock
CFG = {
    'window': 12,
    'num_parts': 64,
    'max_epochs': 15,
    'patience': 4,
    'lr': 3e-4,
    'weight_decay': 5e-4,
    'spatial_hidden_dim': 96,
    'temporal_hidden_dim': 96,
    'gnn_layers': 2,
    'gru_layers': 1,
    'dropout': 0.15,
    'lambda_max': 0.12,
    'warmup_epochs': 12,
    'val_hours': 14 * 24,
    'test_hours': 14 * 24,
    'train_time_stride': 4,
    'eval_time_stride': 4,
    'max_train_windows_per_cluster': 24,
    'max_eval_windows_per_cluster': 24
}

BASELINE = {
    'test_scaled_mse': 0.0009470369910732621,
    'test_mae_unscaled': 5.080196380615234,
    'test_rmse_unscaled': 6.913161277770996
}

print('CFG:', CFG)
print('BASELINE:', BASELINE)

CFG: {'window': 12, 'num_parts': 64, 'max_epochs': 15, 'patience': 4, 'lr': 0.0003, 'weight_decay': 0.0005, 'spatial_hidden_dim': 96, 'temporal_hidden_dim': 96, 'gnn_layers': 2, 'gru_layers': 1, 'dropout': 0.15, 'lambda_max': 0.12, 'warmup_epochs': 12, 'val_hours': 336, 'test_hours': 336, 'train_time_stride': 4, 'eval_time_stride': 4, 'max_train_windows_per_cluster': 24, 'max_eval_windows_per_cluster': 24}
BASELINE: {'test_scaled_mse': 0.0009470369910732621, 'test_mae_unscaled': 5.080196380615234, 'test_rmse_unscaled': 6.913161277770996}


In [3]:
# PHASE 2 - Load graph, coords, and tabular data
graph_path = root / 'data/processed/graph/topology_graph_pyg_inference.pt'
model_input_path = root / 'data/processed/model_input/model_input_node_hourly_features.parquet'
node_map_path = root / 'data/processed/graph/topology_nodeid_to_index_map.parquet'
nodes_path = root / 'data/graphs/bangalore_utm_nodes.parquet'
tensor_dir = root / 'data/processed/tensors'

pyg = torch.load(graph_path, weights_only=False)
edge_index_global = pyg.edge_index.long().cpu()
edge_attr_global = pyg.edge_attr.float().cpu() if hasattr(pyg, 'edge_attr') and pyg.edge_attr is not None else torch.ones(edge_index_global.shape[1], 1, dtype=torch.float32)
num_nodes = int(pyg.num_nodes)

if hasattr(pyg, 'train_mask') and pyg.train_mask is not None:
    train_mask_global = pyg.train_mask.bool().cpu()
else:
    train_mask_global = torch.load(tensor_dir / 'train_mask.pt').bool().cpu()

upwind_path = tensor_dir / 'upwind_edge_mask.pt'
upwind_edge_mask_global = torch.load(upwind_path).bool().cpu() if upwind_path.exists() else None

df = pd.read_parquet(model_input_path)
node_map = pd.read_parquet(node_map_path)
nodes_df = pd.read_parquet(nodes_path)

node_to_idx = dict(zip(node_map['node_id'].values, node_map['node_index'].values))

print({
    'num_nodes': num_nodes,
    'num_edges': int(edge_index_global.shape[1]),
    'train_mask_active': int(train_mask_global.sum().item()),
    'physics_mask_present': upwind_edge_mask_global is not None,
    'model_input_rows': int(len(df))
})

{'num_nodes': 154902, 'num_edges': 424550, 'train_mask_active': 23, 'physics_mask_present': True, 'model_input_rows': 1932700}


In [4]:
# PHASE 3 - Spatial K-Means partitioning and local subgraphs
feature_cols = [
    'station_pm10', 'station_pm25', 'station_no2', 'station_so2', 'station_co',
    'weather_wind_speed_10m', 'weather_wind_direction_10m', 'weather_wind_gusts_10m',
    'weather_temperature_2m', 'weather_relative_humidity_2m', 'weather_surface_pressure',
    'city_nitrogen_dioxide', 'city_sulphur_dioxide', 'city_pm2_5', 'city_pm10', 'city_carbon_monoxide'
]
target_col = 'station_pm25'

feature_cols = [c for c in feature_cols if c in df.columns]
df = df[['timestamp', 'node_id'] + feature_cols + [target_col]].copy()
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp', 'node_id']).copy()
df['node_id'] = pd.to_numeric(df['node_id'], errors='coerce').astype('Int64').dropna().astype('int64')
df['node_index'] = df['node_id'].map(node_to_idx)
df = df.dropna(subset=['node_index']).copy()
df['node_index'] = df['node_index'].astype('int64')
df = df[(df['node_index'] >= 0) & (df['node_index'] < num_nodes)].copy()

coords_df = node_map.merge(
    nodes_df[['osmid', 'x', 'y']],
    left_on='node_id',
    right_on='osmid',
    how='inner'
).sort_values('node_index')

if len(coords_df) != num_nodes:
    raise ValueError('Coordinate join did not cover every node in the graph')

coords = coords_df[['x', 'y']].to_numpy(dtype='float32')

def dominant_wind_dir_deg(frame):
    if 'weather_wind_direction_10m' in frame.columns:
        wd = pd.to_numeric(frame['weather_wind_direction_10m'], errors='coerce').dropna()
        if len(wd) > 0:
            r = np.deg2rad(wd.to_numpy(dtype='float64'))
            s = np.sin(r).mean()
            c = np.cos(r).mean()
            return float((np.rad2deg(np.arctan2(s, c)) + 360.0) % 360.0)
    return 0.0

wind_dir_deg = dominant_wind_dir_deg(df)
NUM_PARTS = 64

print(f'Performing spatial K-Means into {NUM_PARTS} contiguous neighborhoods...')
kmeans = KMeans(n_clusters=NUM_PARTS, random_state=SEED, n_init=10)
cluster_labels = kmeans.fit_predict(coords)

def build_upwind_mask(local_node_ids, local_edge_index, coords_array, wind_dir, threshold_deg=110.0):
    mask = np.zeros(local_edge_index.shape[1], dtype=bool)
    for i in range(local_edge_index.shape[1]):
        s, d = local_edge_index[0, i], local_edge_index[1, i]
        s_g, d_g = local_node_ids[s], local_node_ids[d]
        sx, sy = coords_array[s_g]
        dx, dy = coords_array[d_g]
        ex, ey = float(dx - sx), float(dy - sy)
        if abs(ex) < 1e-8 and abs(ey) < 1e-8: continue
        edge_bearing = (np.degrees(np.arctan2(ey, ex)) + 360.0) % 360.0
        delta = abs(((edge_bearing - wind_dir + 180.0) % 360.0) - 180.0)
        mask[i] = bool(delta >= threshold_deg)
    return torch.from_numpy(mask)

class SpatialPartition:
    def __init__(self, cid, node_ids, edge_index, edge_attr, train_mask, upwind_edge_mask):
        self.cid = int(cid)
        self.n_id = torch.as_tensor(node_ids, dtype=torch.long)
        self.edge_index = edge_index
        self.edge_attr = edge_attr
        self.train_mask = train_mask
        self.upwind_edge_mask = upwind_edge_mask

cluster_data = []
edge_index_np = edge_index_global.numpy()
edge_attr_np = edge_attr_global.numpy()

for cid in range(NUM_PARTS):
    node_ids = np.where(cluster_labels == cid)[0].astype(np.int64)
    if len(node_ids) == 0: continue
    keep = np.isin(edge_index_np[0], node_ids) & np.isin(edge_index_np[1], node_ids)
    local_edge_global = edge_index_np[:, keep]
    if local_edge_global.shape[1] == 0: continue
    local_map = {g: i for i, g in enumerate(node_ids.tolist())}
    re_src = np.array([local_map[int(x)] for x in local_edge_global[0]], dtype=np.int64)
    re_dst = np.array([local_map[int(x)] for x in local_edge_global[1]], dtype=np.int64)
    local_edge_index = torch.tensor(np.stack([re_src, re_dst], axis=0), dtype=torch.long)
    cluster_data.append(SpatialPartition(cid, node_ids, local_edge_index, torch.tensor(edge_attr_np[keep], dtype=torch.float32),
                                        train_mask_global[torch.as_tensor(node_ids, dtype=torch.long)].bool(),
                                        build_upwind_mask(node_ids, local_edge_index, coords, wind_dir_deg)))

print({'num_parts_spatial': len(cluster_data), 'wind_dir_deg': round(float(wind_dir_deg), 2)})

Performing spatial K-Means into 64 contiguous neighborhoods...
{'num_parts_spatial': 64, 'wind_dir_deg': 84.99}


In [5]:
# PHASE 4 - Strict temporal split and cluster helpers (safe replacement)
WINDOW = 12
VAL_HOURS = int(14 * 24)
TEST_HOURS = int(14 * 24)

all_times = pd.Index(sorted(df['timestamp'].unique()))
T_total = len(all_times)
test_start = T_total - TEST_HOURS
val_start = test_start - VAL_HOURS
train_end = val_start

if train_end <= WINDOW + 24:
    raise RuntimeError('Not enough history for strict split')

mask_indices = torch.where(train_mask_global)[0].tolist()
y_sensor_raw = df[df['node_index'].isin(set(mask_indices))][target_col]

if isinstance(y_sensor_raw, pd.DataFrame):
    y_sensor = y_sensor_raw.iloc[:, 0].astype('float32')
else:
    y_sensor = y_sensor_raw.astype('float32')

q95_val = y_sensor.quantile(0.95)
if isinstance(q95_val, pd.Series):
    q95_val = q95_val.iloc[0]

TARGET_SCALE = float(max(50.0, float(q95_val) if pd.notna(q95_val) else 50.0))

raw_target_full = df[target_col].iloc[:, 0] if isinstance(df[target_col], pd.DataFrame) else df[target_col]
df['target_scaled'] = pd.to_numeric(raw_target_full, errors='coerce').astype('float32') / TARGET_SCALE

print({
    'train_end_t': int(train_end),
    'val_start_t': int(val_start),
    'test_start_t': int(test_start),
    'window': int(WINDOW),
    'target_scale_q95': float(TARGET_SCALE)
})

def lambda_phys_schedule(epoch, warmup_epochs=12, max_lambda=0.12):
    if epoch < warmup_epochs:
        return max_lambda * float(epoch + 1) / float(warmup_epochs)
    return max_lambda

def cluster_frame(node_ids, t0, t1):
    times = all_times[t0:t1]
    node_set = set(int(x) for x in node_ids.tolist())
    return df[(df['timestamp'].isin(times)) & (df['node_index'].isin(node_set))].copy()

def _fill_matrix(sub_df, times, local_nodes, value_col):
    time_to_row = {ts: i for i, ts in enumerate(times)}
    node_to_col = {int(n): j for j, n in enumerate(local_nodes.tolist())}
    mat = np.full((len(times), len(local_nodes)), np.nan, dtype='float32')

    if sub_df.empty:
        return mat

    # Groupby handles duplicate labels by averaging them before filling the matrix
    grp = sub_df.groupby(['timestamp', 'node_index'], as_index=False)[value_col].mean()
    for r in grp.itertuples(index=False):
        ts = r[0]
        ni = int(r[1])
        val = r[2]
        if ts in time_to_row and ni in node_to_col:
            mat[time_to_row[ts], node_to_col[ni]] = np.float32(val)

    return mat

def build_cluster_arrays(node_ids, t0, t1):
    sub_df = cluster_frame(node_ids, t0, t1)
    if sub_df.empty:
        return None, None

    local_nodes = np.array(sorted(node_ids.tolist()), dtype=np.int64)
    times = all_times[t0:t1]
    feat_mats = []

    for c in feature_cols:
        mat = _fill_matrix(sub_df, times, local_nodes, c)
        # Apply temporal interpolation on the filled matrix
        mat = pd.DataFrame(mat).ffill(axis=0).bfill(axis=0).fillna(0.0).to_numpy(dtype='float32')
        feat_mats.append(mat)

    x = np.stack(feat_mats, axis=-1)

    y_mat = _fill_matrix(sub_df, times, local_nodes, 'target_scaled')
    y_mat = pd.DataFrame(y_mat).ffill(axis=0).bfill(axis=0).fillna(0.0).to_numpy(dtype='float32')

    return torch.tensor(x, dtype=torch.float32), torch.tensor(y_mat, dtype=torch.float32)

def make_windows(x_tnf, y_tn, window, max_windows=None, stride=1):
    t_max = x_tnf.shape[0]
    starts = list(range(0, t_max - window, stride))
    if len(starts) == 0:
        return None, None
    if max_windows is not None and len(starts) > max_windows:
        starts = random.sample(starts, max_windows)
        starts.sort()

    xs, ys = [], []
    for s in starts:
        xs.append(x_tnf[s:s + window])
        ys.append(y_tn[s + window])

    if len(xs) == 0:
        return None, None
    return torch.stack(xs, dim=0), torch.stack(ys, dim=0)

@torch.no_grad()
def eval_spatial(model, split_t0, split_t1, eval_stride=4, max_windows_per_cluster=24):
    model.eval()
    scaled_losses = []
    abs_err = []
    sq_err = []

    for part in cluster_data:
        node_ids = part.n_id.cpu().long()
        local_mask = part.train_mask.bool()
        if int(local_mask.sum().item()) == 0:
            continue

        x_tnf, y_tn = build_cluster_arrays(node_ids, split_t0, split_t1)
        if x_tnf is None:
            continue

        xb, yb = make_windows(x_tnf, y_tn, WINDOW, max_windows=max_windows_per_cluster, stride=eval_stride)
        if xb is None:
            continue

        edge_i = part.edge_index.to(device)
        edge_a = part.edge_attr.to(device)
        mask_local = local_mask.to(device)

        pred = model(x_seq=xb.to(device), edge_index=edge_i, edge_attr=edge_a)
        loss = masked_data_loss(pred=pred, target=yb.to(device), train_mask=mask_local)
        scaled_losses.append(float(loss.detach().cpu().item()))

        mb = mask_local.view(1, -1).expand(pred.shape[0], -1)
        diff = (pred[mb] - yb.to(device)[mb]).detach().cpu()
        abs_err.append(torch.abs(diff) * TARGET_SCALE)
        sq_err.append((diff * TARGET_SCALE) ** 2)

    if len(scaled_losses) == 0:
        return float('nan'), float('nan'), float('nan')

    mean_scaled = float(np.mean(scaled_losses))
    ae = torch.cat(abs_err).mean().item()
    rmse = torch.sqrt(torch.cat(sq_err).mean()).item()
    return mean_scaled, float(ae), float(rmse)

print('Helpers ready')

{'train_end_t': 87178, 'val_start_t': 87178, 'test_start_t': 87514, 'window': 12, 'target_scale_q95': 342.93560791015625}
Helpers ready


In [6]:
# PHASE 4 - NumPy flattening and DataFrame teardown
import gc
import numpy as np
import pandas as pd

if 'df' not in globals():
    raise RuntimeError('df is missing. Run the data load cell first.')
if 'feature_cols' not in globals() or 'TARGET_SCALE' not in globals():
    raise RuntimeError('feature_cols or TARGET_SCALE missing. Run the temporal prep cell first.')

# Keep only the fields needed for cache building.
keep_cols = ['timestamp', 'node_id', 'node_index'] + feature_cols + ['target_scaled']
keep_cols = [c for c in keep_cols if c in df.columns]
df = df[keep_cols].copy()

all_times = pd.Index(sorted(df['timestamp'].unique()))
T_total = len(all_times)
time_codes = pd.Categorical(df['timestamp'], categories=all_times, ordered=True).codes.astype(np.int32)
raw_ts_indices = np.ascontiguousarray(time_codes)
raw_node_indices = np.ascontiguousarray(df['node_index'].to_numpy(dtype=np.int32, copy=True))

# Feature columns + target_scaled in one contiguous float32 array.
raw_data_values = np.ascontiguousarray(
    df[feature_cols + ['target_scaled']].to_numpy(dtype=np.float32, copy=True)
)

# Sort once so time slices are contiguous and cheap.
sort_order = np.lexsort((raw_node_indices, raw_ts_indices))
raw_data_values = raw_data_values[sort_order]
raw_ts_indices = raw_ts_indices[sort_order]
raw_node_indices = raw_node_indices[sort_order]

# Precompute slice boundaries for each timestamp index.
time_breaks = np.searchsorted(raw_ts_indices, np.arange(T_total + 1, dtype=np.int32), side='left')

# Drop the DataFrame now to reclaim RAM before the cache loop.
del df
gc.collect()

print({
    'T_total': int(T_total),
    'raw_rows': int(raw_data_values.shape[0]),
    'feature_dim': int(len(feature_cols)),
    'raw_data_values_shape': tuple(raw_data_values.shape),
    'raw_ts_indices_shape': tuple(raw_ts_indices.shape),
    'raw_node_indices_shape': tuple(raw_node_indices.shape)
})

{'T_total': 87850, 'raw_rows': 1932700, 'feature_dim': 16, 'raw_data_values_shape': (1932700, 18), 'raw_ts_indices_shape': (1932700,), 'raw_node_indices_shape': (1932700,)}
